# 🔬 Master EDA Laboratory
*Dataset: `{{FILE_NAME}}`*

This notebook contains the complete pipeline for Exploratory Data Analysis, Outlier Detection, Scaling, Feature Selection, and Clustering based on your standard methodology.

**INSTRUCTIONS:** Read through the cells, modify the `TARGET_COL` where requested, and run the pipeline.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, MaxAbsScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, RFE

# System path for database uploader
sys.path.append('../../') 
from scripts.db_uploader import upload_to_postgres

In [ ]:
# 1. LOAD DATA
df = pd.read_csv('../../data/raw/{{FILE_NAME}}', low_memory=False)
print(f'✅ Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(10)

## Phase 1: Basic Exploration & Imputation

In [ ]:
# Data overview
df.info()
print('\n--- Missing Values ---')
print(df.isnull().sum())

# Statistical summary
display(df.describe().round(2))

# Clean Empty Structures
df.dropna(how='all', axis=1, inplace=True)
df.dropna(how='all', axis=0, inplace=True)
df.fillna(0, inplace=True) # Basic imputation (customize as needed)

## Phase 2: Encoding & Datetime Parsing

In [ ]:
# Auto-detect and parse datetime columns
for col in df.columns:
    if 'date' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col])
            print(f'✅ Converted {col} to datetime.')
        except:
            pass

# Categorical Variables Encoding
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()
for col in cat_cols:
    unique_vals = [str(x).lower() for x in df[col].unique()]
    if 'true' in unique_vals or 'false' in unique_vals:
        df[col] = df[col].map({True: 1, False: 0, 'True': 1, 'False': 0, '1': 1, '0': 0}).fillna(0).astype(int)
    else:
        df[col] = df[col].astype(str)
        df[col+'_encoded'] = LabelEncoder().fit_transform(df[col])
print('✅ Encoding completed.')

## Phase 3: Visualizations & Correlation

In [ ]:
numerical_data = df.select_dtypes(include=np.number)

# 3.1 Data Distribution (Histograms)
print('--- Data Distributions ---')
num_cols_to_plot = min(5, len(numerical_data.columns)) # Plot first 5 numeric cols to avoid clutter
for col in numerical_data.columns[:num_cols_to_plot]:
    plt.figure(figsize=(6, 3))
    sns.histplot(df[col], kde=True, bins=30, color='skyblue')
    plt.title(f'Distribution of {col}')
    plt.show()

# 3.2 Correlation Matrix Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(numerical_data.corr(), annot=False, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

## Phase 4: Outlier Detection & Treatment

In [ ]:
# 4.1 Visual Outlier Detection (Boxplots)
print('--- Outlier Boxplots ---')
for col in numerical_data.columns[:num_cols_to_plot]:
    plt.figure(figsize=(6, 2))
    sns.boxplot(x=df[col], color='lightgreen')
    plt.title(f'Boxplot of {col}')
    plt.show()

# 4.2 Outlier Treatment Functions
def remove_outliers_tukey(data, column, k=1.5):
    """Removes rows with outliers based on IQR"""
    if column not in data.columns: return data
    q1 = np.percentile(data[column].dropna(), 25)
    q3 = np.percentile(data[column].dropna(), 75)
    iqr = q3 - q1
    lower_fence = q1 - k * iqr
    upper_fence = q3 + k * iqr
    return data[(data[column] >= lower_fence) & (data[column] <= upper_fence)]

def winsorize_outliers(data, column, lower_percentile=0.05, upper_percentile=0.95):
    """Caps extreme values without deleting rows"""
    if column not in data.columns: return data
    upper_lim = data[column].quantile(upper_percentile)
    lower_lim = data[column].quantile(lower_percentile)
    data.loc[(data[column] > upper_lim), column] = upper_lim
    data.loc[(data[column] < lower_lim), column] = lower_lim
    return data

# Example Usage (Uncomment to apply):
# df = remove_outliers_tukey(df, 'price')
# df = winsorize_outliers(df, 'points')
print('✅ Outlier functions loaded.')

## Phase 5: Scaling & Normalization

In [ ]:
# Select columns to scale (excluding IDs or Target if necessary)
cols_to_scale = numerical_data.columns

# Standard Scaling (Mean=0, Variance=1)
std_scaler = StandardScaler()
# df[cols_to_scale] = std_scaler.fit_transform(df[cols_to_scale])

# Min-Max Scaling (Range 0 to 1)
mm_scaler = MinMaxScaler()
# df[cols_to_scale] = mm_scaler.fit_transform(df[cols_to_scale])

print('✅ Scalers initialized. Uncomment the desired scaler line to apply it.')

## Phase 6: Feature Selection & Clustering

In [ ]:
# --- CONFIGURATION ---
# IMPORTANT: Set your main target column here for Feature Selection analysis
TARGET_COL = 'your_target_column_here' 

In [ ]:
print('--- Machine Learning Tools ---')

# 6.1 Feature Importance (Tree-based)
if TARGET_COL in df.columns:
    try:
        X = df.select_dtypes(include=np.number).drop(TARGET_COL, axis=1).fillna(0)
        y = df[TARGET_COL].fillna(0)
        model = RandomForestClassifier(n_estimators=50, random_state=42)
        model.fit(X, y)
        
        feat_importances = pd.Series(model.feature_importances_, index=X.columns)
        plt.figure(figsize=(10, 5))
        feat_importances.nlargest(10).plot(kind='barh', color='teal')
        plt.title('Top 10 Most Important Features')
        plt.show()
    except Exception as e:
        print(f'Feature Importance skipped: {e}')
else:
    print(f'⚠️ Target column "{TARGET_COL}" not found. Update configuration in Cell 1 to run Feature Selection.')

# 6.2 K-Means Clustering Example
# Example: Clustering users/players based on 2 numeric columns
def apply_kmeans(data, cols_for_clustering, num_clusters=3):
    if all(c in data.columns for c in cols_for_clustering):
        kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
        data['cluster_id'] = kmeans.fit_predict(data[cols_for_clustering].fillna(0))
        print(f'✅ K-Means applied. Created {num_clusters} clusters.')
        return data
    return data

# df = apply_kmeans(df, ['numeric_col_1', 'numeric_col_2'], num_clusters=3)

## 🚀 Phase 7: Finish EDA & Upload to Database
Execute this cell to upload the polished DataFrame to your PostgreSQL (Silver Layer).

In [ ]:
# Ensure the database connection script has the correct password configured in your .env file
upload_to_postgres(df, table_name='{{TABLE_NAME}}')